In [ ]:
import pandas as pd
import numpy as np
import os
import torch
import joblib 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

input_file = r'D:\disertatie\CIC_IDS_2017\Tuesday-WorkingHours.pcap_ISCX.csv'
output_path = r'D:\disertatie\preprocesare_anomalii_2017'
os.makedirs(output_path, exist_ok=True)

print("--- PROCESARE CIC-IDS-2017 (Semi-Supervised) ---")

df = pd.read_csv(input_file)
df.columns = df.columns.str.strip()

features_audit = [
    'Destination Port',        
    'Flow Packets/s',          
    'Flow IAT Mean',           
    'Init_Win_bytes_forward',  
    'Init_Win_bytes_backward', 
    'Flow Duration',           
    'Total Fwd Packets',       
    'Total Backward Packets',  
    'Packet Length Mean'       
]

X = df[features_audit].copy()

X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.median(), inplace=True)

# Generam masca binara din eticheta text originala
df['Label_Bin'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

print(f" Distributie initiala: Normal: {sum(df['Label_Bin']==0)} | Atacuri: {sum(df['Label_Bin']==1)}")

# Separam datele brute pe clase
X_normal = X[df['Label_Bin'] == 0].values
X_anomalies = X[df['Label_Bin'] == 1].values

# Impartim datele normale: 80% antrenare, 20% pastrate pentru testare
X_train_normal, X_test_normal = train_test_split(X_normal, test_size=0.2, random_state=42)

# Setul de test final = restul de 20% date normale + TOATE atacurile
X_test_final = np.vstack([X_test_normal, X_anomalies])

y_train_final = np.zeros(len(X_train_normal))
y_test_final = np.concatenate([np.zeros(len(X_test_normal)), np.ones(len(X_anomalies))])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_normal)
X_test_scaled = scaler.transform(X_test_final)

print(f" Date partajate: Train_Normal={len(X_train_scaled)}, Test_Total={len(X_test_scaled)} (Normal={len(X_test_normal)}, Atac={len(X_anomalies)})")
print(f" Salvare fisiere .pt si scaler in {output_path}...")

torch.save(torch.tensor(X_train_scaled, dtype=torch.float32), os.path.join(output_path, 'X_train_anomalii.pt'))
torch.save(torch.tensor(y_train_final, dtype=torch.float32), os.path.join(output_path, 'y_train_anomalii.pt'))

torch.save(torch.tensor(X_test_scaled, dtype=torch.float32), os.path.join(output_path, 'X_test_anomalii.pt'))
torch.save(torch.tensor(y_test_final, dtype=torch.float32), os.path.join(output_path, 'y_test_anomalii.pt'))

joblib.dump(scaler, os.path.join(output_path, 'scaler_anomalii.pkl'))


--- PROCESARE CIC-IDS-2017 (Semi-Supervised) ---
 Distributie initiala: Normal: 432074 | Atacuri: 13835
 Date partajate: Train_Normal=345659, Test_Total=100250 (Normal=86415, Atac=13835)
 Salvare fisiere .pt si scaler in D:\disertatie\preprocesare_anomalii_2017...
------------------------------
PREPROCESARE FINALIZATA CU SUCCES (ALINIATA CU 2018)!
